In [1]:
#importing the Libraies
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
dataset = pd.read_csv("final_data.csv")

In [3]:
dataset = pd.get_dummies(dataset, dtype=int,drop_first='True')

In [4]:
dataset

,age,experience_years_total,experience_years_in_field,num_skills,annual_salary_usd,has_equity,equity_vesting_years,vacation_weeks,overtime_hours_per_month,months_since_last_promotion,...,certifications_TensorFlow Developer Certificate; PMP,certifications_TensorFlow Developer Certificate; PMP; CEH,certifications_TensorFlow Developer Certificate; PMP; CISSP,certifications_TensorFlow Developer Certificate; PMP; Certified Scrum Master,certifications_TensorFlow Developer Certificate; PMP; CompTIA Security+,certifications_TensorFlow Developer Certificate; PMP; Kubernetes CKA,company_size_Large (201-1000),company_size_Medium (51-200),company_size_Small (11-50),company_size_Startup (1-10)
0,32.0,7.7,3.2,3.0,100290.66,0.0,0.0,3.0,12.9,38.0,...,0,0,0,0,0,0,0,0,1,0
1,30.0,3.0,1.9,8.0,92383.12,1.0,1.0,4.0,6.5,26.0,...,0,0,0,0,0,0,0,0,0,1
2,32.0,0.5,0.5,8.0,107153.59,1.0,4.0,5.0,13.2,0.0,...,0,0,0,0,0,0,0,0,0,0
3,31.0,14.5,12.4,4.0,217074.54,1.0,1.0,2.0,5.9,111.0,...,0,0,0,0,0,0,0,1,0,0
4,42.0,7.9,3.8,4.0,152246.34,0.0,0.0,6.0,0.0,20.0,...,0,0,0,0,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,38.0,6.2,3.9,6.0,218700.43,1.0,4.0,3.0,7.3,52.0,...,0,0,0,0,0,0,1,0,0,0
9996,37.0,9.7,7.0,6.0,170835.76,0.0,0.0,5.0,8.4,58.0,...,0,0,0,0,0,0,0,0,1,0
9997,31.0,0.5,0.5,5.0,25695.15,1.0,3.0,4.0,17.1,0.0,...,0,0,0,0,0,0,0,0,0,1
9998,32.0,8.7,8.3,8.0,264531.10,0.0,0.0,4.0,14.3,78.0,...,0,0,0,0,0,0,0,0,0,0


In [5]:
indep_X = dataset.drop(columns=['is_actively_looking'])
dep_Y = dataset['is_actively_looking']

In [6]:
#split into training set and test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(indep_X, dep_Y , test_size = 1/3, random_state = 0)

In [7]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [8]:
from sklearn.linear_model import LogisticRegression

In [9]:
from sklearn.model_selection import GridSearchCV

param_grid = {'solver':['newton-cg', 'lbfgs', 'liblinear', 'saga'],
             'penalty':['l2']} 



grid = GridSearchCV(LogisticRegression(), param_grid, refit = True, verbose = 3,n_jobs=-1,scoring='f1_weighted') 
   
# fitting the model for grid search 
grid.fit(X_train, y_train) 

Fitting 5 folds for each of 4 candidates, totalling 20 fits


,estimator,LogisticRegression()
,param_grid,"{'penalty': ['l2'], 'solver': ['newton-cg', 'lbfgs', ...]}"
,scoring,'f1_weighted'
,n_jobs,-1
,refit,True
,cv,None
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,penalty,'l2'


In [10]:
# print best parameter after tuning 
#print(grid.best_params_) 
re=grid.cv_results_
#print(re)
grid_predictions = grid.predict(X_test) 
   

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, grid_predictions)



# print classification report 
from sklearn.metrics import classification_report
clf_report = classification_report(y_test, grid_predictions)

In [11]:
from sklearn.metrics import f1_score
f1_macro=f1_score(y_test,grid_predictions,average='weighted')
print("The f1_macro value for best parameter {}:".format(grid.best_params_),f1_macro)

The f1_macro value for best parameter {'penalty': 'l2', 'solver': 'newton-cg'}: 0.7184750895239846


In [12]:
print("The confusion Matrix:\n",cm)

The confusion Matrix:
 [[2680    5]
 [ 648    1]]


In [13]:
print("The report:\n",clf_report)

The report:
               precision    recall  f1-score   support

         0.0       0.81      1.00      0.89      2685
         1.0       0.17      0.00      0.00       649

    accuracy                           0.80      3334
   macro avg       0.49      0.50      0.45      3334
weighted avg       0.68      0.80      0.72      3334



In [14]:
from sklearn.metrics import roc_auc_score
roc_auc_score(y_test,grid.predict_proba(X_test)[:,1])

0.501764353123126

In [15]:
table=pd.DataFrame.from_dict(re)

In [16]:
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_penalty,param_solver,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,133.883892,11.210189,1.992526,0.166191,l2,newton-cg,"{'penalty': 'l2', 'solver': 'newton-cg'}",0.705726,0.706551,0.706551,0.706551,0.705515,0.706179,0.000460,1
1,65.038139,20.862464,2.355293,2.920347,l2,lbfgs,"{'penalty': 'l2', 'solver': 'lbfgs'}",0.654339,0.655924,0.686078,0.706551,0.676587,0.675896,0.019532,2
2,75.874722,17.908960,1.096072,0.147342,l2,liblinear,"{'penalty': 'l2', 'solver': 'liblinear'}",0.071931,0.070026,0.074413,0.076160,0.076625,0.073831,0.002516,3
3,439.366349,14.007261,0.152922,0.032918,l2,saga,"{'penalty': 'l2', 'solver': 'saga'}",0.071931,0.070026,0.072883,0.077686,0.076406,0.073786,0.002845,4


In [18]:
age_input=float(input("Age:"))
experience_years_total_input=float(input("Experience:"))
Skills_input=float(input("No.of skills:"))
sex_male_input=int(input("Sex Male 0 or 1:"))
annual_salary_usd_input=float(input("Annual Salary(USD):"))

Age: 38
Experience: 6.2
No.of skills: 5
Sex Male 0 or 1: 0
Annual Salary(USD): 218700.43


In [21]:
Future_Prediction=grid.predict([[age_input,experience_years_total_input,Skills_input,sex_male_input,annual_salary_usd_input]])# change the paramter,play with it.
print("Future_Prediction={}".format(Future_Prediction))

ValueError: X has 5 features, but LogisticRegression is expecting 22352 features as input.